In [1]:
# Notebook 04 - Recommendation Model (step-by-step)
#
# Input: artifacts/features/ from Notebook 03
# Output: artifacts/models/ (SVD model, hybrid config, evaluation results)
#
# Cell 1 - Setup & Load Artifacts
#   Load all feature artifacts, build lookup dicts, verify shapes
#
# Cell 2 - Popularity-Based Recommender
#   Rank by Bayesian popularity_score, fallback for cold-start users
#
# Cell 3 - Content-Based Recommender
#   Similarity lookup from pre-computed top-k neighbors
#   User personalization via high-rated recipe neighbors
#
# Cell 4 - Collaborative Filtering (SVD)
#   User-item sparse matrix -> TruncatedSVD -> predicted ratings
#   Only works for users seen in train
#
# Cell 5 - Constraint Filter
#   Rule-based filter: max_calories, max_minutes, tags_include
#   Wraps around any recommender output
#
# Cell 6 - Hybrid Combiner
#   Weighted: alpha * content_score + (1-alpha) * cf_score
#   Fallback tiers: cold -> popularity, warm -> content, active -> hybrid
#
# Cell 7 - Qualitative Check
#   Spot-check 3-5 sample users (active / warm / cold-start)
#
# Cell 8 - Evaluation on Validation Set
#   Precision@K, Recall@K, NDCG@K, Coverage
#   Cold-start user report
#
# Cell 9 - Save Model Artifacts
#   svd_model.joblib, hybrid_config.json, evaluation_results.json

In [2]:
# === Cell 1 - Setup & Load Artifacts ===

import pandas as pd
import numpy as np
import ast
import json
import joblib
from pathlib import Path
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from scipy.sparse import csr_matrix

FEATURES_DIR = Path("../artifacts/features")

# ---------- Load artifacts ----------

with open(FEATURES_DIR / "feature_manifest.json") as f:
    manifest = json.load(f)

tfidf_vectorizer = joblib.load(FEATURES_DIR / "tfidf_vectorizer.joblib")
recipe_tfidf_matrix = sparse.load_npz(FEATURES_DIR / "recipe_tfidf_matrix.npz")

recipe_index_map = pd.read_csv(FEATURES_DIR / "recipe_index_map.csv")
recipe_similarity_topk = pd.read_csv(FEATURES_DIR / "recipe_similarity_topk.csv")
recipe_features = pd.read_csv(FEATURES_DIR / "recipe_features.csv")
popularity_features = pd.read_csv(FEATURES_DIR / "popularity_features.csv")

user_features = pd.read_csv(FEATURES_DIR / "user_features.csv")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

train_interactions = pd.read_csv(FEATURES_DIR / "train_interactions.csv")
val_interactions = pd.read_csv(FEATURES_DIR / "val_interactions.csv")
train_interactions["date"] = pd.to_datetime(train_interactions["date"], errors="coerce")
val_interactions["date"] = pd.to_datetime(val_interactions["date"], errors="coerce")

# ---------- Load recipe names from processed data ----------

df_recipes = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "name"])
recipe_name_map = dict(zip(df_recipes["id"], df_recipes["name"]))

# ---------- Build lookup dicts ----------

id2idx = dict(zip(recipe_index_map["recipe_id"], recipe_index_map["matrix_index"]))
idx2id = dict(zip(recipe_index_map["matrix_index"], recipe_index_map["recipe_id"]))

train_user_ids = set(train_interactions["user_id"].unique())
train_recipe_ids = set(train_interactions["recipe_id"].unique())

# ---------- Verify shapes ----------

print(f"Manifest version:        {manifest['version']}")
print(f"Build timestamp:         {manifest['build_timestamp']}")
print()
print(f"recipe_tfidf_matrix:     {recipe_tfidf_matrix.shape}")
print(f"recipe_index_map:        {len(recipe_index_map):,} rows")
print(f"recipe_similarity_topk:  {len(recipe_similarity_topk):,} rows")
print(f"recipe_features:         {len(recipe_features):,} rows")
print(f"popularity_features:     {len(popularity_features):,} rows")
print(f"user_features:           {len(user_features):,} rows")
print(f"train_interactions:      {len(train_interactions):,} rows")
print(f"val_interactions:        {len(val_interactions):,} rows")
print()
print(f"Unique train users:      {len(train_user_ids):,}")
print(f"Unique train recipes:    {len(train_recipe_ids):,}")
print(f"id2idx entries:          {len(id2idx):,}")
print(f"recipe_name_map entries: {len(recipe_name_map):,}")
print()
print(f"Sample user_features favorite_tags: {user_features['favorite_tags'].iloc[0]}")
print("\nAll artifacts loaded successfully.")

Manifest version:        1.0
Build timestamp:         2026-04-25T20:01:56.461237

recipe_tfidf_matrix:     (76608, 20000)
recipe_index_map:        77,300 rows
recipe_similarity_topk:  3,865,000 rows
recipe_features:         77,300 rows
popularity_features:     77,300 rows
user_features:           27,021 rows
train_interactions:      559,500 rows
val_interactions:        139,876 rows

Unique train users:      27,021
Unique train recipes:    72,860
id2idx entries:          77,300
recipe_name_map entries: 76,608

Sample user_features favorite_tags: ['time-to-make', 'preparation', 'course']

All artifacts loaded successfully.


In [3]:
# === Cell 2 - Popularity-Based Recommender ===
#
# Xếp hạng recipe theo popularity_score (Bayesian smoothing từ NB03).
# Đây là recommender đơn giản nhất, dùng làm:
#   - Baseline so sánh với các model phức tạp hơn
#   - Fallback cho cold-start users (không có lịch sử)

# Pre-sort popularity một lần, dùng lại cho mọi lần gọi
popularity_ranked = popularity_features.sort_values("popularity_score", ascending=False).reset_index(drop=True)

def recommend_popular(n=10, exclude_ids=None):
    """Return top-n popular recipes, optionally excluding already-seen IDs."""
    candidates = popularity_ranked
    if exclude_ids:
        candidates = candidates[~candidates["recipe_id"].isin(exclude_ids)]
    top = candidates.head(n).copy()
    top["rank"] = range(1, len(top) + 1)
    top["name"] = top["recipe_id"].map(recipe_name_map)
    return top[["rank", "recipe_id", "name", "popularity_score"]].reset_index(drop=True)

# ---------- Quick test ----------
print("Top-10 popular recipes:")
display(recommend_popular(10))

Top-10 popular recipes:


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216
5,6,58516,hot pepper jelly,4.961464
6,7,244193,absolutely the best new york cheesecake gluten...,4.959104
7,8,55309,caprese salad tomatoes italian marinated tomatoes,4.958252
8,9,31639,7 layer bean dip,4.957364
9,10,25094,my chicken parmigiana,4.957087


In [4]:
# === Cell 3 - Content-Based Recommender ===
# - recommend_content_similar(recipe_id, n): lookup pre-computed top-k neighbors
# - recommend_for_user_content(user_id, n):
#     Get recipes user rated >= 4 in train
#     Aggregate neighbors, remove already seen, rank by avg similarity
#     Fallback: use favorite_tags or popular if no history

# Build neighbor lookup
neighbors_lookup = recipe_similarity_topk.groupby("recipe_id")


# -------------------------------------------------
# Recommend recipes similar to a given recipe
# -------------------------------------------------

def recommend_content_similar(recipe_id, n=10):

    if recipe_id not in neighbors_lookup.groups:
        return pd.DataFrame()

    neighbors = neighbors_lookup.get_group(recipe_id)

    neighbors = neighbors.sort_values("similarity", ascending=False)

    top = neighbors.head(n).copy()

    top["name"] = top["neighbor_recipe_id"].map(recipe_name_map)
    top["rank"] = range(1, len(top) + 1)

    return top.rename(
        columns={"neighbor_recipe_id": "recipe_id"}
    )[["rank", "recipe_id", "name", "similarity"]]


# -------------------------------------------------
# Content recommendation for a user
# -------------------------------------------------

def recommend_for_user_content(user_id, n=10):

    user_history = train_interactions[
        train_interactions["user_id"] == user_id
    ]

    liked = user_history[user_history["rating"] >= 4]["recipe_id"].tolist()

    # ---------- fallback: cold user ----------
    if len(liked) == 0:

        user_row = user_features[user_features["user_id"] == user_id]

        if len(user_row) == 0:
            return recommend_popular(n)

        fav_tags = user_row.iloc[0]["favorite_tags"]

        if len(fav_tags) == 0:
            return recommend_popular(n)

        tag_recipes = df_recipes[
            df_recipes["id"].isin(
                recipe_features["recipe_id"]
            )
        ]

        candidates = popularity_features.sort_values(
            "popularity_score", ascending=False
        )

        return recommend_popular(n)

    # ---------- aggregate neighbors ----------
    scores = defaultdict(list)

    for recipe_id in liked:

        if recipe_id not in neighbors_lookup.groups:
            continue

        neighbors = neighbors_lookup.get_group(recipe_id)

        for _, row in neighbors.iterrows():
            scores[row["neighbor_recipe_id"]].append(row["similarity"])

    seen = set(user_history["recipe_id"])

    scored = {
        r: np.mean(vals)
        for r, vals in scores.items()
        if r not in seen
    }

    ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)[:n]

    result = pd.DataFrame(ranked, columns=["recipe_id", "content_score"])

    result["name"] = result["recipe_id"].map(recipe_name_map)
    result["rank"] = range(1, len(result) + 1)

    return result[["rank", "recipe_id", "name", "content_score"]]

In [5]:
# === Cell 4 - Collaborative Filtering (SVD) ===
# - Build user-item sparse matrix from train_interactions (centered by user mean)
# - Fit TruncatedSVD(n_components=50)
# - recommend_cf(user_id, n): reconstruct predicted ratings, exclude seen, return top-N
# - Only works for users present in train set

# Build user-item matrix
# -------------------------------------------------

user_ids = train_interactions["user_id"].unique()
recipe_ids = train_interactions["recipe_id"].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
item_to_index = {r: i for i, r in enumerate(recipe_ids)}

index_to_user = {i: u for u, i in user_to_index.items()}
index_to_item = {i: r for r, i in item_to_index.items()}

rows = train_interactions["user_id"].map(user_to_index)
cols = train_interactions["recipe_id"].map(item_to_index)
data = train_interactions["rating"]

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(recipe_ids))
)

print("User-item matrix shape:", user_item_matrix.shape)


# -------------------------------------------------
# Center ratings by user mean
# -------------------------------------------------

user_means = train_interactions.groupby("user_id")["rating"].mean()

centered_data = train_interactions.apply(
    lambda r: r["rating"] - user_means[r["user_id"]],
    axis=1
)

centered_matrix = csr_matrix(
    (centered_data, (rows, cols)),
    shape=user_item_matrix.shape
)


# -------------------------------------------------
# Train SVD
# -------------------------------------------------

svd_model = TruncatedSVD(
    n_components=50,
    random_state=42
)

user_factors = svd_model.fit_transform(centered_matrix)
item_factors = svd_model.components_

print("SVD model trained")


# -------------------------------------------------
# CF recommendation function
# -------------------------------------------------

item_ids_array = np.array(recipe_ids)
user_seen_lookup = train_interactions.groupby("user_id")["recipe_id"].apply(set).to_dict()

def recommend_cf(user_id, n=10):

    if user_id not in user_to_index:
        return recommend_popular(n)

    u_idx = user_to_index[user_id]
    user_vector = user_factors[u_idx]

    scores = user_vector @ item_factors
    scores = scores + user_means[user_id]

    seen = user_seen_lookup.get(user_id, set())
    if seen:
        candidate_mask = ~np.isin(item_ids_array, list(seen))
        candidate_indices = np.where(candidate_mask)[0]
    else:
        candidate_indices = np.arange(len(item_ids_array))

    if len(candidate_indices) == 0:
        return pd.DataFrame(columns=["rank", "recipe_id", "name", "predicted_rating"])

    candidate_scores = scores[candidate_indices]
    top_k = min(n, len(candidate_indices))

    if top_k == len(candidate_indices):
        top_local = np.argsort(-candidate_scores)
    else:
        top_local = np.argpartition(-candidate_scores, top_k - 1)[:top_k]
        top_local = top_local[np.argsort(-candidate_scores[top_local])]

    top_indices = candidate_indices[top_local]
    ranked_ids = item_ids_array[top_indices]
    ranked_scores = candidate_scores[top_local]

    result = pd.DataFrame({
        "recipe_id": ranked_ids,
        "predicted_rating": ranked_scores,
    })

    result["name"] = result["recipe_id"].map(recipe_name_map)
    result["rank"] = range(1, len(result) + 1)

    return result[["rank", "recipe_id", "name", "predicted_rating"]]

User-item matrix shape: (27021, 72860)
SVD model trained


In [6]:
# === Cell 5 - Constraint Filter ===
# - apply_constraints(candidate_ids, max_calories, max_minutes, tags_include)
# - Filters candidate recipes by rule-based conditions
# - Wraps around any recommender output before returning to user
# Load tags cho tag filter (nếu chưa có)
df_recipes_full = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "tags_clean"])
df_recipes_full["tags_clean"] = df_recipes_full["tags_clean"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
def apply_constraints(candidate_df, max_calories=None, max_minutes=None,
                      tags_include=None, max_ingredients=None):
    """Lọc output của bất kỳ recommender nào theo ràng buộc người dùng."""
    df = candidate_df.merge(
        recipe_features[["recipe_id", "calories", "minutes", "n_ingredients_calc"]],
        on="recipe_id", how="left"
    )
    if max_calories is not None:
        df = df[df["calories"] <= max_calories]
    if max_minutes is not None:
        df = df[df["minutes"] <= max_minutes]
    if max_ingredients is not None:
        df = df[df["n_ingredients_calc"] <= max_ingredients]
    if tags_include is not None:
        tags_set = set(tags_include)
        tags_map = df_recipes_full.set_index("id")["tags_clean"]
        df = df[df["recipe_id"].map(tags_map).apply(
            lambda t: tags_set.issubset(set(t)) if isinstance(t, list) else False
        )]
    df = df.drop(columns=["calories", "minutes", "n_ingredients_calc"], errors="ignore")
    df = df.reset_index(drop=True)
    df["rank"] = range(1, len(df) + 1)
    return df



In [7]:
# === Cell 6 - Hybrid Combiner ===
# - hybrid_score = alpha * content_score_norm + (1-alpha) * cf_score_norm
# - Fallback tiers:
#     Cold-start (no train history)  -> Popularity + Content from tags
#     Warm (< 5 interactions)        -> Content-heavy (alpha=0.7)
#     Active (>= 5 interactions)     -> Balanced hybrid (alpha=0.5)
# - recommend_hybrid(user_id, n, alpha, constraints)
# - Returns results with source tag: popularity / content / cf / hybrid

def min_max_normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)
def recommend_hybrid(user_id, n=10, constraints=None):
    seen_ids = set(
        train_interactions[train_interactions["user_id"] == user_id]["recipe_id"]
    )
    # ════════════════════════════════════════
    # TIER 1: COLD-START
    # ════════════════════════════════════════
    if user_id not in train_user_ids:
        results = recommend_popular(n=n * 3, exclude_ids=seen_ids)
        results["source"] = "popularity"
        if constraints:
            results = apply_constraints(results, **constraints)
        return results.head(n).reset_index(drop=True)
    user_row = user_features[user_features["user_id"] == user_id]
    if user_row.empty:
        results = recommend_popular(n=n * 3, exclude_ids=seen_ids)
        results["source"] = "popularity"
        if constraints:
            results = apply_constraints(results, **constraints)
        return results.head(n).reset_index(drop=True)
    rating_count = user_row["rating_count"].values[0]
    # ════════════════════════════════════════
    # TIER 2: WARM (< 5 ratings) → α = 0.7
    # TIER 3: ACTIVE (≥ 5)       → α = 0.5
    # ════════════════════════════════════════
    alpha = 0.7 if rating_count < 5 else 0.5
    content_results = recommend_for_user_content(user_id, n=n * 5)
    cf_results = recommend_cf(user_id, n=n * 5)
    # ─── Cả 2 rỗng ───
    if content_results.empty and cf_results.empty:
        results = recommend_popular(n=n * 3, exclude_ids=seen_ids)
        results["source"] = "popularity"
        if constraints:
            results = apply_constraints(results, **constraints)
        return results.head(n).reset_index(drop=True)
    # ─── Kiểm tra kết quả thực hay fallback ───
    has_content = "content_score" in content_results.columns
    has_cf = "predicted_rating" in cf_results.columns
    # CF rỗng hoặc Content là fallback → dùng cái còn lại
    if not has_content and not has_cf:
        results = recommend_popular(n=n * 3, exclude_ids=seen_ids)
        results["source"] = "popularity"
        if constraints:
            results = apply_constraints(results, **constraints)
        return results.head(n).reset_index(drop=True)
    if not has_content:
        cf_results["source"] = "cf"
        if constraints:
            cf_results = apply_constraints(cf_results, **constraints)
        return cf_results.head(n).reset_index(drop=True)
    if not has_cf:
        content_results["source"] = "content"
        if constraints:
            content_results = apply_constraints(content_results, **constraints)
        return content_results.head(n).reset_index(drop=True)
    # ─── Cả 2 đều có kết quả thực → MERGE + COMBINE ───
    content_df = content_results[["recipe_id", "content_score"]]
    cf_df = cf_results[["recipe_id", "predicted_rating"]].rename(
        columns={"predicted_rating": "cf_score"}
    )
    merged = content_df.merge(cf_df, on="recipe_id", how="outer")
    merged["content_score"] = merged["content_score"].fillna(merged["content_score"].median())
    merged["cf_score"] = merged["cf_score"].fillna(merged["cf_score"].median())
    merged["content_norm"] = min_max_normalize(merged["content_score"])
    merged["cf_norm"] = min_max_normalize(merged["cf_score"])
    merged["hybrid_score"] = alpha * merged["content_norm"] + (1 - alpha) * merged["cf_norm"]
    merged = merged.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
    merged["rank"] = range(1, len(merged) + 1)
    merged["name"] = merged["recipe_id"].map(recipe_name_map)
    merged["source"] = "hybrid"
    result = merged[["rank", "recipe_id", "name", "hybrid_score",
                      "content_norm", "cf_norm", "source"]]
    if constraints:
        result = apply_constraints(result, **constraints)
    return result.head(n).reset_index(drop=True)



In [8]:
# === Cell 7 - Qualitative Check ===
# - Pick 3-5 sample users: 1 active, 1 warm, 1 cold-start
# - Print recommendations from each model + hybrid
# - Compare: recipe name, score, source
# ─── Chọn 1 user mỗi loại ───
sample_active = user_features[user_features["activity_level"] == "active"]["user_id"].iloc[0]
sample_warm = user_features[user_features["activity_level"] == "warm"]["user_id"].iloc[0]
cold_users = set(val_interactions["user_id"]) - train_user_ids
sample_cold = list(cold_users)[0]
sample_users = [
    (sample_active, "ACTIVE"),
    (sample_warm, "WARM"),
    (sample_cold, "COLD-START"),
]
# ─── So sánh tất cả models cho mỗi user ───
for user_id, tier_label in sample_users:
    print("=" * 70)
    print(f"USER {user_id} ({tier_label})")
    print("=" * 70)
    # Lịch sử user (nếu có)
    history = train_interactions[train_interactions["user_id"] == user_id]
    liked = history[history["rating"] >= 4]
    print(f"Total interactions: {len(history)}")
    print(f"Liked (rating ≥ 4): {len(liked)}")
    if not liked.empty:
        liked_names = liked["recipe_id"].head(5).map(recipe_name_map).tolist()
        print(f"Sample liked: {liked_names}")
    print()
    # 1) Popularity
    print("── Popularity ──")
    seen = set(history["recipe_id"]) if not history.empty else None
    display(recommend_popular(5, exclude_ids=seen))
    print()
    # 2) Content
    print("── Content-Based ──")
    display(recommend_for_user_content(user_id, n=5))
    print()
    # 3) CF
    print("── Collaborative Filtering ──")
    cf_result = recommend_cf(user_id, n=5)
    if cf_result.empty:
        print("(empty — user không có trong train)")
    else:
        display(cf_result)
    print()
    # 4) Hybrid
    print("── Hybrid ──")
    display(recommend_hybrid(user_id, n=5))
    print("\n")

USER 1533 (ACTIVE)
Total interactions: 73
Liked (rating ≥ 4): 71
Sample liked: ['zucchini and rice casserole', 'orange yogurt cream', 'parmesan fish in the oven', 'fennel mashed potatoes', 'c hudson s ham potato casserole']

── Popularity ──


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



── Content-Based ──


,rank,recipe_id,name,content_score
0,1,46277.0,no salt seasoning,0.702102
1,2,68017.0,italian seasoning,0.597112
2,3,4029.0,herb blend for boursin cream cheese,0.583898
3,4,48555.0,memphis rub,0.569698
4,5,83902.0,montreal steak seasoning,0.551078



── Collaborative Filtering ──


...

,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



── Collaborative Filtering ──


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



── Hybrid ──


,rank,recipe_id,name,popularity_score,source
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171,popularity
1,2,34753,basic sourdough bread,4.967149,popularity
2,3,154351,kittencal s balsamic vinaigrette,4.965450,popularity
3,4,78579,pan release professional pan coating better th...,4.965094,popularity
4,5,186029,the best creole cajun seasoning mix,4.964216,popularity


In [11]:
# === Cell 8 - Evaluation on Validation Set ===
# - Ground truth: recipes rated >= 4 per user in val_interactions
# - Predicted: top-K from each model
# - Metrics: Precision@K, Recall@K, NDCG@K, Coverage
# - Comparison table: Popularity vs Content vs CF vs Hybrid
# - Separate cold-start user report
import time
from collections import defaultdict
K = 10
# --- Ground truth: only evaluate unseen positives in validation ---
val_positive = val_interactions[val_interactions["rating"] >= 4].copy()
train_seen_by_user = train_interactions.groupby("user_id")["recipe_id"].apply(set).to_dict()

def build_unseen_relevant(group):
    user_id = group.name
    seen = train_seen_by_user.get(user_id, set())
    return set(group[~group.isin(seen)])

val_relevant = (
    val_positive.groupby("user_id")["recipe_id"]
    .apply(build_unseen_relevant)
    .to_dict()
)
val_relevant = {u: items for u, items in val_relevant.items() if len(items) > 0}
val_user_ids = list(val_relevant.keys())
print(f"Val users with unseen relevant items: {len(val_user_ids):,}")
print(f"Avg unseen relevant items per user: {np.mean([len(v) for v in val_relevant.values()]):.1f}")
# ─── Metric functions ───
def precision_at_k(recommended, relevant, k):
    rec_set = set(recommended[:k])
    return len(rec_set & relevant) / k if k > 0 else 0.0
def recall_at_k(recommended, relevant, k):
    rec_set = set(recommended[:k])
    return len(rec_set & relevant) / len(relevant) if len(relevant) > 0 else 0.0
def ndcg_at_k(recommended, relevant, k):
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)   # i+2 vì log2(1) = 0
    # Ideal DCG: tất cả relevant items nằm đầu
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0
# ─── Evaluate 1 model ───
def evaluate_model(recommend_fn, model_name, user_ids, k=K, max_error_rate=0.01):
    precisions = []
    recalls = []
    ndcgs = []
    all_recommended = set()
    skipped = 0
    errors = 0
    t0 = time.time()

    for i, user_id in enumerate(user_ids):
        relevant = val_relevant.get(user_id, set())
        if len(relevant) == 0:
            skipped += 1
            continue

        try:
            result = recommend_fn(user_id, n=k)
            if not isinstance(result, pd.DataFrame) or "recipe_id" not in result.columns:
                raise ValueError("recommend_fn must return DataFrame with 'recipe_id' column")
            rec_list = result["recipe_id"].tolist()
        except Exception as exc:
            errors += 1
            rec_list = []
            if errors <= 5:
                print(f"  [{model_name}] ERROR user={user_id}: {exc}")

        precisions.append(precision_at_k(rec_list, relevant, k))
        recalls.append(recall_at_k(rec_list, relevant, k))
        ndcgs.append(ndcg_at_k(rec_list, relevant, k))
        all_recommended.update(rec_list)

        if (i + 1) % 2000 == 0:
            elapsed = time.time() - t0
            print(f"  [{model_name}] {i+1:,}/{len(user_ids):,} users ({elapsed:.0f}s), errors={errors}")

    elapsed = time.time() - t0
    users_evaluated = len(precisions)
    error_rate = (errors / users_evaluated) if users_evaluated > 0 else 0.0
    if error_rate > max_error_rate:
        raise RuntimeError(
            f"[{model_name}] Evaluation error rate too high: {error_rate:.2%} > {max_error_rate:.2%}"
        )

    n_total_items = len(recipe_name_map)
    coverage = len(all_recommended) / n_total_items if n_total_items > 0 else 0
    metrics = {
        "model": model_name,
        "P@K": np.mean(precisions) if precisions else 0,
        "R@K": np.mean(recalls) if recalls else 0,
        "NDCG@K": np.mean(ndcgs) if ndcgs else 0,
        "Coverage": coverage,
        "users_evaluated": users_evaluated,
        "users_skipped": skipped,
        "errors": errors,
        "error_rate": round(error_rate, 6),
        "time_seconds": round(elapsed, 1),
    }
    print(f"  [{model_name}] Done in {elapsed:.1f}s - P@{k}={metrics['P@K']:.4f}, errors={errors}")
    return metrics
# ─── Chạy evaluation cho từng model ───
print(f"\n{'='*60}")
print(f"Evaluating on {len(val_user_ids):,} val users, K={K}")
print(f"{'='*60}\n")
results = []

# Adapter to keep a consistent interface recommend_fn(user_id, n)
def recommend_popularity_for_user(user_id, n=10):
    seen_ids = train_seen_by_user.get(user_id, set())
    return recommend_popular(n=n, exclude_ids=seen_ids)

# 1) Popularity
print("▶ Popularity")
results.append(evaluate_model(recommend_popularity_for_user, "Popularity", val_user_ids))
# 2) Content
print("\n▶ Content-Based")
results.append(evaluate_model(recommend_for_user_content, "Content", val_user_ids))
# 3) CF
print("\n▶ Collaborative Filtering")
results.append(evaluate_model(recommend_cf, "CF (SVD)", val_user_ids))
# 4) Hybrid
print("\n▶ Hybrid")
results.append(evaluate_model(recommend_hybrid, "Hybrid", val_user_ids))
# ─── Bảng so sánh ───
print(f"\n{'='*60}")
print(f"COMPARISON TABLE (K={K})")
print(f"{'='*60}\n")
comparison = pd.DataFrame(results)
comparison = comparison[["model", "P@K", "R@K", "NDCG@K", "Coverage",
                          "users_evaluated", "time_seconds"]]
display(comparison)
# ─── Cold-start report riêng ───
print(f"\n{'='*60}")
print("COLD-START USER REPORT")
print(f"{'='*60}\n")
cold_start_val_users = [u for u in val_user_ids if u not in train_user_ids]
print(f"Cold-start users in val with relevant items: {len(cold_start_val_users):,}\n")
if cold_start_val_users:
    cold_results = []
    for name, fn in [("Popularity", recommend_popularity_for_user),
                     ("Content", recommend_for_user_content),
                     ("Hybrid", recommend_hybrid)]: 
        print(f"▶ {name} (cold-start only)")
        cold_results.append(evaluate_model(fn, name, cold_start_val_users))
    cold_comparison = pd.DataFrame(cold_results)
    cold_comparison = cold_comparison[["model", "P@K", "R@K", "NDCG@K", "Coverage"]]
    print()
    display(cold_comparison)

Val users with unseen relevant items: 14,129
Avg unseen relevant items per user: 9.4

Evaluating on 14,129 val users, K=10

▶ Popularity
  [Popularity] 2,000/14,129 users (124s), errors=0
  [Popularity] 4,000/14,129 users (235s), errors=0
  [Popularity] 6,000/14,129 users (344s), errors=0
  [Popularity] 8,000/14,129 users (444s), errors=0
  [Popularity] 10,000/14,129 users (542s), errors=0
  [Popularity] 12,000/14,129 users (634s), errors=0
  [Popularity] 14,000/14,129 users (724s), errors=0
  [Popularity] Done in 729.8s - P@10=0.0009, errors=0

▶ Content-Based
  [Content] 2,000/14,129 users (515s), errors=0
  [Content] 4,000/14,129 users (954s), errors=0
  [Content] 6,000/14,129 users (1342s), errors=0
  [Content] 8,000/14,129 users (1657s), errors=0
  [Content] 10,000/14,129 users (1903s), errors=0
  [Content] 12,000/14,129 users (2108s), errors=0
  [Content] 14,000/14,129 users (2318s), errors=0
  [Content] Done in 2329.1s - P@10=0.0006, errors=0

▶ Collaborative Filtering
  [CF (SV

,model,P@K,R@K,NDCG@K,Coverage,users_evaluated,time_seconds
0,Popularity,0.000878,0.001366,0.001254,0.000196,14129,729.8
1,Content,0.000602,0.000808,0.000842,0.329874,14129,2329.1
2,CF (SVD),0.002350,0.004176,0.004037,0.017152,14129,840.4
3,Hybrid,0.001600,0.002880,0.002694,0.306926,14129,1857.1



COLD-START USER REPORT

Cold-start users in val with relevant items: 3,787

▶ Popularity (cold-start only)
  [Popularity] 2,000/3,787 users (58s), errors=0
  [Popularity] Done in 109.3s - P@10=0.0007, errors=0
▶ Content (cold-start only)
  [Content] 2,000/3,787 users (62s), errors=0
  [Content] Done in 116.2s - P@10=0.0007, errors=0
▶ Hybrid (cold-start only)
  [Hybrid] 2,000/3,787 users (60s), errors=0
  [Hybrid] Done in 115.0s - P@10=0.0007, errors=0



,model,P@K,R@K,NDCG@K,Coverage
0,Popularity,0.000713,0.001096,0.000879,0.000131
1,Content,0.000713,0.001096,0.000879,0.000131
2,Hybrid,0.000713,0.001096,0.000879,0.000131


In [10]:
# === Cell 9 - Save Model Artifacts ===
# - Save to artifacts/models/:
#     svd_model.joblib
#     user_index_map.csv
#     hybrid_config.json (alpha, tiers, constraint defaults)
#     evaluation_results.json
import json
import joblib
from pathlib import Path
from datetime import datetime
MODELS_DIR = Path("../artifacts/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
# ─── 1. SVD Model ───
svd_artifact = {
    "svd_model": svd_model,
    "user_factors": user_factors,
    "item_factors": item_factors,
    "user_means": user_means.to_dict(),
}
joblib.dump(svd_artifact, MODELS_DIR / "svd_model.joblib")
print(f"Saved: svd_model.joblib ({(MODELS_DIR / 'svd_model.joblib').stat().st_size / 1024 / 1024:.1f} MB)")
# ─── 2. User Index Map ───
user_index_df = pd.DataFrame([
    {"user_id": uid, "matrix_index": idx}
    for uid, idx in user_to_index.items()
])
user_index_df.to_csv(MODELS_DIR / "user_index_map.csv", index=False)
print(f"Saved: user_index_map.csv ({len(user_index_df):,} users)")
# ─── 3. Hybrid Config ───
hybrid_config = {
    "version": "1.0",
    "build_timestamp": datetime.now().isoformat(),
    "svd_n_components": 50,
    "tiers": {
        "cold_start": {
            "condition": "user not in train",
            "strategy": "popularity"
        },
        "warm": {
            "condition": "rating_count < 5",
            "alpha": 0.7,
            "strategy": "content-heavy hybrid"
        },
        "active": {
            "condition": "rating_count >= 5",
            "alpha": 0.5,
            "strategy": "balanced hybrid"
        }
    },
    "constraint_defaults": {
        "max_calories": None,
        "max_minutes": None,
        "tags_include": None,
        "max_ingredients": None
    },
    "data_stats": {
        "train_users": len(train_user_ids),
        "train_interactions": len(train_interactions),
        "val_interactions": len(val_interactions),
        "total_recipes": len(recipe_name_map),
    }
}
with open(MODELS_DIR / "hybrid_config.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_config, f, indent=2, ensure_ascii=False)
print(f"Saved: hybrid_config.json")
# ─── 4. Evaluation Results ───
# `results` là list of dicts từ Cell 8
evaluation_output = {
    "K": K,
    "timestamp": datetime.now().isoformat(),
    "all_users": results,
}
# Thêm cold-start results nếu có
if "cold_results" in dir() and cold_results:
    evaluation_output["cold_start_users"] = cold_results

with open(MODELS_DIR / "evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_output, f, indent=2, ensure_ascii=False)
print(f"Saved: evaluation_results.json")
# ─── Summary ───
print(f"\nAll artifacts saved to: {MODELS_DIR.resolve()}")
for f in sorted(MODELS_DIR.iterdir()):
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<35s} {size:>8.2f} MB")
print(f"\nNotebook 04 COMPLETE.")


Saved: svd_model.joblib (66.3 MB)
Saved: user_index_map.csv (27,021 users)
Saved: hybrid_config.json
Saved: evaluation_results.json

All artifacts saved to: D:\GITHUB\Food-RecommendationSystem\artifacts\models
  evaluation_results.json                 0.00 MB
  hybrid_config.json                      0.00 MB
  svd_model.joblib                       66.25 MB
  user_index_map.csv                      0.35 MB

Notebook 04 COMPLETE.
